# 3장 1강 · 객체 탐지 및 세그멘테이션 평가

**실습문제**

사진 한 장에서 개의 종류를 맞히고도 머리 박스는 틀릴 수 있습니다. 또 머리 위치가 어긋난 사진에서도 몸 전체의 윤곽은 잘 나올 수 있습니다. 이 실습에서는 같은 Oxford 사진을 세 가지 질문으로 평가합니다.

| 구분 | 제출할 결과 |
|---|---|
| 필수 1 | 전처리와 학습 기록, 머리 박스의 면적 검산, IoU 기준에 따른 판정 |
| 필수 2 | 중복·클래스 오류 판정표, 실제 예측의 Precision·Recall, 클래스별 AP와 mAP |
| 독립 과제 | 4×4 예제의 위치 수정, 실제 몸 마스크의 픽셀 오차 그림, 사진별 평균과 검사 범위에 대한 결론 |

**데이터와 실행 준비**　기존 `oxford_dataset(1).zip` 또는 `oxford_dataset.zip`을 노트북 옆에 둡니다. 이 수업용 ZIP에는 사진과 머리 좌표가 있고, 몸 마스크 파일은 없습니다. 첫 실행 시 Oxford 공식 픽셀 주석 `annotations.tar.gz`(약 19 MB)와 ImageNet 사전학습 ResNet18 가중치(약 45 MB)를 내려받습니다. 이미 받은 주석은 노트북 옆에 두면 됩니다. 주석의 값 1은 동물, 2는 배경, 3은 미분류 영역이며 3은 학습 손실과 평가에서 모두 제외합니다. [Oxford-IIIT Pet 공식 주석](https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz)

**실행 환경**　Python 3.10 이상, PyTorch·torchvision, NumPy, pandas, Pillow와 Jupyter가 필요합니다. 제작 환경은 CPU, torch 2.6.0, torchvision 0.21.0입니다. 시드 42를 고정했고, 환경에 따라 소수점 아래 수치가 조금 다를 수 있습니다.

**실험 조건**　사진마다 주석된 머리가 하나인 경우만 사용합니다. 제공본에서 JPEG·머리 주석을 함께 쓸 수 있는 사진 중 고양이·개 각각 학습 96장, 평가 24장을 선택합니다. 출력은 사진당 종 한 개, 머리 박스 한 개, 몸 전체의 전경 마스크 한 장입니다. 점수는 두 종 중 선택한 종의 확률이므로 박스가 정확할 확률까지 나타내지는 않습니다. 여러 객체가 있는 장면의 정식 COCO 성능과는 평가 조건이 다릅니다.

**실행 순서**　공통 준비 셀을 위에서 아래로 실행하면 두 파일 모두 실제로 모델을 학습해 같은 예측 표를 만듭니다. 이후 필수 1·필수 2·독립 과제 중 원하는 절부터 시작할 수 있습니다. 연습용 고정 후보와 4×4 배열은 손계산용이며, 모델 성능 표는 평가 사진 48장의 실제 출력으로 계산합니다.

### 공통 준비 1 · 파일 확인과 사진 분할

In [1]:
from pathlib import Path
from io import BytesIO
import tarfile
import hashlib
from urllib.request import urlretrieve
from zipfile import ZipFile
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as T

torch.set_num_threads(2)
SEED, SIZE, MASK_SIZE = 42, 160, 40
species = ["cat", "dog"]
coords = ["xmin","ymin","xmax","ymax"]
data_path = next((p for p in [Path("oxford_dataset(1).zip"), Path("oxford_dataset.zip")] if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("노트북 옆에 수업용 oxford_dataset.zip 또는 oxford_dataset(1).zip을 두세요.")
trimap_file = Path("annotations.tar.gz")
if not trimap_file.is_file():
    temp_file = trimap_file.with_suffix(".download")
    urlretrieve("https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz", temp_file)
    if hashlib.sha256(temp_file.read_bytes()).hexdigest() != "52425fb6de5c424942b7626b428656fcbd798db970a937df61750c0f1d358e91":
        raise ValueError("다운로드한 Oxford 주석 파일의 내용이 다릅니다.")
    temp_file.replace(trimap_file)

with ZipFile(data_path) as z:
    boxes = pd.read_csv(z.open("head_boxes.csv"))
    present = set(z.namelist())
pool = boxes.loc[boxes.image_path.isin(present) & boxes.object_name.isin(species)].copy()
pool = pool[~pool.image_id.duplicated(keep=False)].set_index("image_id").sort_index()
# 예제로 읽는 사진 3장은 학습 후보에서 먼저 제외합니다.
demo_ids = ["Abyssinian_1","beagle_110","beagle_111"]
train_ids, eval_ids = [], []
for name in species:
    fixed = [i for i in demo_ids if pool.loc[i,"object_name"] == name]
    candidates = pool.loc[pool.object_name == name].drop(index=fixed)
    order = candidates.sample(frac=1,random_state=SEED).index.tolist()
    train_ids.extend(order[:96])
    eval_ids.extend(fixed + order[96:96+24-len(fixed)])
assert len(set(train_ids)) == 192 and len(set(eval_ids)) == 48
assert set(train_ids).isdisjoint(eval_ids)
print("가용 단일 머리 주석 사진",len(pool)," / 학습",len(train_ids),"/ 평가",len(eval_ids))

가용 단일 머리 주석 사진 735  / 학습 192 / 평가 48


### 공통 준비 2 · 원본 사진, 머리 박스, 몸 마스크의 크기 맞추기

In [2]:
all_ids = train_ids + eval_ids
wanted = {f"annotations/trimaps/{key}.png": key for key in all_ids}
trimaps = {}
with tarfile.open(trimap_file, "r|gz") as tar:
    for entry in tar:
        if entry.name in wanted and entry.isfile():
            with Image.open(BytesIO(tar.extractfile(entry).read())) as im:
                trimaps[wanted[entry.name]] = np.array(im)
assert len(trimaps) == len(all_ids)

def load_photo(key):
    with ZipFile(data_path) as z, Image.open(BytesIO(z.read(pool.loc[key,"image_path"]))) as im:
        return im.convert("RGB")

def prepare_sample(im, key):
    row = pool.loc[key]
    w,h = im.size
    rgb = im.resize((SIZE,SIZE), Image.Resampling.BILINEAR)
    x = T.normalize(T.to_tensor(rgb), [0.485,0.456,0.406], [0.229,0.224,0.225])
    norm_box = row[coords].to_numpy(dtype=np.float32) / np.array([w,h,w,h], dtype=np.float32)
    orig = trimaps[key]
    assert orig.shape == (h,w)
    mask = np.array(Image.fromarray(orig).resize((MASK_SIZE,MASK_SIZE), Image.Resampling.NEAREST))
    return x, species.index(row.object_name), norm_box, mask

def extract_inputs():
    weights = ResNet18_Weights.DEFAULT
    base = resnet18(weights=weights).eval()
    backbone = nn.Sequential(*list(base.children())[:7]).eval()
    feats, labels, targets, masks = [], [], [], []
    with torch.no_grad(), ZipFile(data_path) as z:
        for start in range(0, len(all_ids), 12):
            rgb_batch = []
            for key in all_ids[start:start+12]:
                with Image.open(BytesIO(z.read(pool.loc[key,"image_path"]))) as im:
                    x,y,b,m = prepare_sample(im.convert("RGB"),key)
                rgb_batch.append(x)
                labels.append(y)
                targets.append(b)
                masks.append(m)
            feats.append(backbone(torch.stack(rgb_batch)))
    return torch.cat(feats), torch.tensor(labels), torch.from_numpy(np.stack(targets)), torch.from_numpy(np.stack(masks))

### 공통 준비 3 · 사진 한 장에서 종·머리 위치·몸 영역을 예측하는 모델

In [3]:
class PetHeads(nn.Module):
    def __init__(self):
        super().__init__()
        self.kind = nn.Linear(256,2)
        self.region = nn.Sequential(nn.AdaptiveAvgPool2d(3), nn.Flatten(), nn.Linear(256*9,64),
                                    nn.ReLU(), nn.Linear(64,4))
        self.body = nn.Sequential(nn.Conv2d(256,32,3,padding=1), nn.ReLU(), nn.Conv2d(32,1,1))

    def forward(self, f):
        cls = self.kind(f.mean(dim=(2,3)))
        center, span = self.region(f).sigmoid().split(2,dim=1)
        xyxy = torch.cat([center-span/2, center+span/2], dim=1).clamp(0,1)
        mask = F.interpolate(self.body(f), size=(MASK_SIZE,MASK_SIZE), mode="bilinear",align_corners=False)
        return cls, xyxy, mask[:,0]

def fit_heads(features, labels, boxes, masks):
    torch.manual_seed(SEED)
    net = PetHeads()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    n = len(train_ids)
    trace = []
    for epoch in range(8):
        totals = np.zeros(3)
        for idx in torch.randperm(n).split(16):
            cls, predicted_box, logits = net(features[idx])
            trimap = masks[idx]
            valid = trimap != 3
            kind_loss = F.cross_entropy(cls,labels[idx])
            box_loss = F.smooth_l1_loss(predicted_box,boxes[idx],beta=0.1)
            mask_loss = F.binary_cross_entropy_with_logits(logits[valid],(trimap[valid]==1).float())
            loss = kind_loss + 4*box_loss + mask_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            totals += len(idx)*np.array([kind_loss.item(),box_loss.item(),mask_loss.item()])
        trace.append(totals/n)
    history = pd.DataFrame(trace, columns=["class_loss","box_loss","mask_loss"],index=range(1,9))
    history.index.name = "epoch"
    return net.eval(), history

### 공통 준비 4 · 학습하고 처음 보는 사진 48장의 예측 보관하기

In [4]:
features, y, head_boxes, body_masks = extract_inputs()
net, history = fit_heads(features, y, head_boxes, body_masks)
display(history.round(4))
print("특징/라벨/머리 박스/몸 마스크",tuple(features.shape),tuple(y.shape),tuple(head_boxes.shape),tuple(body_masks.shape))

with torch.no_grad():
    scores, norm_boxes, mask_logits = net(features[len(train_ids):])
    top_score, top_label = scores.softmax(dim=1).max(dim=1)
    body_prob = mask_logits.sigmoid().numpy()
records = []
for i,key in enumerate(eval_ids):
    w,h = load_photo(key).size
    box = norm_boxes[i].numpy() * [w,h,w,h]
    records.append((key, species[top_label[i].item()],top_score[i].item(),tuple(box)))
predictions = pd.DataFrame(records,columns=["image","label","score","box"])
gt = pool.loc[eval_ids]
display(predictions.drop(columns="box").head())

특징/라벨/머리 박스/몸 마스크 (240, 256, 10, 10) (240,) (240, 4) (240, 40, 40)


,class_loss,box_loss,mask_loss
epoch,,,
1,0.6955,0.0785,0.4876
2,0.6860,0.0549,0.2445
3,0.6783,0.0413,0.1764
4,0.6712,0.0323,0.1514
5,0.6654,0.0259,0.1392
6,0.6592,0.0209,0.1309
7,0.6513,0.0177,0.1265
8,0.6448,0.0151,0.1227


,image,label,score
0,Abyssinian_1,cat,0.542233
1,Birman_116,cat,0.526019
2,Egyptian_Mau_12,cat,0.564239
3,Persian_156,cat,0.507792
4,British_Shorthair_102,cat,0.541080


### 공통 준비 5 · 문항에서 호출할 평가 함수와 손계산 자료

In [5]:
def box_parts(a,b):
    left, top = max(a[0],b[0]), max(a[1],b[1])
    right, bottom = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,right-left) * max(0,bottom-top)
    area_a = (a[2]-a[0])*(a[3]-a[1])
    area_b = (b[2]-b[0])*(b[3]-b[1])
    union = area_a + area_b - inter
    return inter,union,inter/union if union else 0.0

def match_detections(gt, candidates, min_iou, min_score=0.0):
    # 평가 표본에는 사진당 정답 머리가 한 개입니다.
    used = set()
    records = []
    kept = candidates[candidates.score >= min_score].sort_values("score", ascending=False,kind="stable")
    for r in kept.itertuples(index=False):
        target = gt.loc[r.image]
        inter, union, iou = box_parts(r.box, target[coords].to_numpy(dtype=float))
        if r.label != target.object_name:
            reason = "클래스 오류"
        elif iou < min_iou:
            reason = "위치 오류"
        elif r.image in used:
            reason = "중복"
        else:
            reason = "TP"
            used.add(r.image)
        records.append((r.image, r.label, r.score, iou, reason, int(reason=="TP")))
    return pd.DataFrame(records,columns=["image","label","score","iou","reason","tp"])

def ap_curve(gt, candidates, name, iou):
    table = match_detections(gt, candidates[candidates.label==name], iou)
    n = int((gt.object_name==name).sum())
    tp = table.tp.cumsum().to_numpy()
    precision = tp / np.arange(1,len(table)+1)
    recall = tp / n
    grid = np.linspace(0,1,101)
    envelope = [precision[recall >= r-1e-12].max(initial=0) for r in grid]
    return float(np.mean(envelope)), pd.DataFrame({"score":table.score,"TP_cum":tp,"precision":precision,"recall":recall})

def mask_counts(trimap, pred):
    valid = trimap != 3
    true = trimap == 1
    tp = int((valid & true & pred).sum())
    fp = int((valid & ~true & pred).sum())
    fn = int((valid & true & ~pred).sum())
    iou = tp / (tp+fp+fn) if tp+fp+fn else np.nan
    dice = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
    return {"TP_px":tp, "FP_px":fp, "FN_px":fn, "mask_iou":iou, "dice":dice}

def show_box(key, proposed):
    canvas = load_photo(key)
    draw = ImageDraw.Draw(canvas)
    draw.rectangle(tuple(pool.loc[key,coords].astype(float)),outline="lime",width=4)
    draw.rectangle(tuple(proposed),outline="darkorange",width=2)
    display(canvas)

In [6]:
# 머리 박스 평가용 가상 검출 목록. A와 B는 같은 사진의 고양이 후보입니다.
toy = pd.DataFrame([
    ("A", "Abyssinian_1", "cat", 0.95, (347, 71, 440, 158)),
    ("B", "Abyssinian_1", "cat", 0.65, (340, 71, 433, 158)),
    ("C", "beagle_110", "cat", 0.60, (70, 33, 338, 269)),
    ("D", "beagle_110", "dog", 0.55, (90, 33, 358, 269)),
], columns=["id", "image", "label", "score", "box"])
# 교안 3절의 Mask. 1은 평가 대상의 픽셀, 0은 그 밖의 픽셀입니다.
true_mask = np.array([[0,0,0,0], [0,1,1,0], [0,1,1,0], [0,0,0,0]])
pred_mask = np.array([[0,0,0,0], [0,0,1,1], [0,0,1,1], [0,0,0,0]])
display(toy)

,id,image,label,score,box
0,A,Abyssinian_1,cat,0.95,"(347, 71, 440, 158)"
1,B,Abyssinian_1,cat,0.65,"(340, 71, 433, 158)"
2,C,beagle_110,cat,0.60,"(70, 33, 338, 269)"
3,D,beagle_110,dog,0.55,"(90, 33, 358, 269)"


## 필수 1 · 실제 모델의 머리 박스를 좌표로 검산하기

### 1-1. 사진·박스·마스크의 좌표계 연결

beagle_111의 원본 크기, RGB 입력·마스크 텐서의 모양, 원본·정규화된 머리 박스를 출력하세요. 정규화된 좌표를 원본 크기로 복원해 주석과 같은지 검사하고 마스크에 최근접 이웃 보간을 쓰는 이유를 설명하세요.

In [ ]:
# 1-1 실습 코드
# 도움말 · prepare_sample의 x, y, b, m과 원본 (width, height)를 사용하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 1-2. 실제 학습 기록과 평가 표본 구분

두 세트의 종별 개수와 공통 ID 수를 확인하세요. 학습 손실표에서 첫·마지막 에포크의 세 손실과 가중합을 비교하고, 무엇이 실제로 학습됐는지 설명하세요.

In [ ]:
# 1-2 실습 코드
# 도움말 · 총손실은 class_loss + 4 × box_loss + mask_loss입니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 1-3. 같은 비글의 두 박스로 교집합과 합집합 구하기

평가 사진 beagle_111의 실제 모델 박스를 정답과 겹쳐 그리세요. 두 왼쪽·위 경계 중 큰 쪽과 오른쪽·아래 경계 중 작은 쪽을 고르고 교집합, 합집합, IoU를 직접 계산하세요. 제공 함수와도 대조하세요.

In [ ]:
# 1-3 실습 코드
# 도움말 · 초록색이 정답이고 주황색이 예측입니다. 좌표 반올림은 마지막 출력에서만 합니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 1-4. 같은 후보의 판정이 기준에 따라 달라지는 이유

beagle_111과 Abyssinian_1에서 예측 종, 종 점수, 박스 IoU, IoU 0.50·0.75의 판정을 출력하세요. 두 사진 중 종을 맞혔지만 0.50에서도 검출에 실패한 사진을 찾고 “점수”와 “위치 정확도”를 구분하세요.

In [ ]:
# 1-4 실습 코드
# 도움말 · 판정에는 종 일치와 IoU 기준 충족이 모두 필요합니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

## 필수 2 · 한 정답을 한 번만 세고 실제 예측으로 AP 계산하기

### 2-1. 두 사진과 네 후보에서 중복·오분류 구별하기

교안 연습 후보 A~D를 점수 순으로 검토하세요. 점수 기준 0.50과 0.80에서 각각 TP·FP·FN을 세고, B의 IoU가 충분한데도 TP가 되지 않는 이유를 쓰세요.

In [ ]:
# 2-1 실습 코드
# 도움말 · 연습 후보는 같은 이미지에 둘 이상 있을 수 있습니다. 각 기준에서 연결 기록을 비우고 시작하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 2-2. 같은 48장에서 점수 기준 세 개 비교하기

실제 모델 결과에 점수 기준 0.50, 0.55, 0.80을 적용하세요. 박스 IoU 기준은 0.50으로 고정하고 검출 수·TP·FP·FN·Precision·Recall을 구하세요. 남은 검출이 0개일 때의 분모도 설명하세요.

In [ ]:
# 2-2 실습 코드
# 도움말 · 일치한 정답 수가 TP입니다. FP의 분모와 FN의 기준 집합을 먼저 적어보세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 2-3. 실제 점수 순위로 클래스별 AP와 mAP 계산하기

고양이의 상위 5개 예측에서 누적 TP·Precision·Recall을 읽으세요. 두 종에 대해 IoU 0.50부터 0.95까지 0.05 간격으로 AP를 구하고 mAP@0.50, mAP@0.50:0.95를 계산하세요.

In [ ]:
# 2-3 실습 코드
# 도움말 · ap_curve는 점수 제한 없이 해당 클래스 예측을 모두 사용합니다. 한 행은 IoU 한 기준이고 한 열은 한 종입니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 2-4. 한 줄의 Precision이 높은데 mAP는 낮을 수 있을까?

2-2와 2-3의 값을 인용해 점수 기준과 IoU 기준을 각각 올리면 무엇이 바뀌는지 서술하세요. Precision 100%와 mAP@0.50 약 0.42가 같은 모델에서 함께 나오는 이유를 쓰세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

## 과제 · 실제 몸 마스크에서 누락·과다 예측을 찾아 결과 해석하기

### 3-1. 4×4 마스크를 움직여 지표의 분모 검산하기

교안 4×4 예시에서 공통·과다·누락 픽셀의 수와 좌표(행, 열)를 출력하고 IoU·Dice를 구하세요. 예측을 왼쪽으로 한 칸 옮겨 오른쪽 끝을 0으로 채운 뒤 같은 지표를 다시 계산하세요.

In [ ]:
# 3-1 실습 코드
# 도움말 · np.roll을 그대로 쓰면 오른쪽 끝으로 픽셀이 되돌아올 수 있습니다. 빈 배열에 필요한 열만 복사하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 3-2. 비글의 몸에서 틀린 9칸은 어디인가?

beagle_111의 종·머리 점수와 구분해서 몸 마스크를 평가하세요. 픽셀 확률 기준 0.50에서 미분류 칸을 제외하고 TP·FP·FN과 IoU·Dice를 직접 계산하세요. 원본·정답·예측·오차 그림을 나란히 표시하세요.

In [ ]:
# 3-2 실습 코드
# 도움말 · 40×40에서 계산하고 시각화할 때만 확대하세요. 오류 그림에서 미분류 픽셀의 색을 따로 두세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 3-3. 픽셀 기준이 높아지면 평균 마스크가 좋아지는가?

동일한 몸 확률맵을 0.50과 0.70에서 이진화하세요. beagle_111과 Abyssinian_1의 과다·누락·두 지표, 전체 48장의 사진별 평균 IoU와 Dice를 비교하세요.

In [ ]:
# 3-3 실습 코드
# 도움말 · 모델을 다시 학습하지 않습니다. 같은 body_prob에서 비교 연산자 오른쪽의 값만 바꿉니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

### 3-4. 찾은 머리의 사진만 남겨도 되는가?

머리 기준(점수 0.50, 박스 IoU 0.50)에서 통과한 사진·실패한 사진·전체의 몸 마스크 평균 IoU와 장수를 비교하세요. 제품이 머리를 찾은 뒤 몸 결과를 보여준다면 어떤 수치를 함께 보고해야 하나요?

In [ ]:
# 3-4 실습 코드
# 도움말 · 사진 ID로 검출 판정과 마스크 지표를 연결하세요. 평가 대상 48장이 모두 결과에 남았는지 확인합니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 실제 출력 또는 계산 근거
- 결과 해석

## 실습 마무리

1. **무엇이 문제였나요?** 분류 성공과 머리 위치 성공이 달랐던 사진을 하나 고르고, 종·좌표·픽셀 가운데 무엇을 비교했는지 쓰세요.
2. **무엇을 바꾸었나요?** 후보 점수 기준, 박스 IoU 판정 기준, 몸 픽셀 기준을 각각 바꿀 때 비교 대상과 분모에 생긴 변화를 쓰세요.
3. **무엇을 근거로 결론을 내렸나요?** 실제 평가 48장의 탐지 지표, 몸 마스크 지표, 원본 사진 한 사례를 인용하고 이 표본으로 확인한 범위를 적으세요.